# Iceberg V3 Geometry on Wherobots — Staging Validation

This notebook exercises the **Iceberg V3 native geometry/geography** support released in havasu-iceberg **v1.11.5**, under the production flag combination:

| Flag | Value | Meaning |
| --- | --- | --- |
| `spark.sql.geospatial.enabled` | `true` | Spark 4.1's `GEOMETRY(srid)` DDL and native geo types are available |
| `spark.sedona.enableNativeGeoTypes` | `false` | the session works in Sedona `GeometryUDT` values (gate **off**) |
| `havasu.geo.v3-write-enabled` (catalog) | `true` | creating V3-native geo columns is permitted on this catalog |

With this combination, **storage is Iceberg V3** (ISO-WKB, Parquet `GEOMETRY` logical type) while the engine works in Sedona UDTs — so every `ST_` function binds, and `GROUP BY`/`DISTINCT` on geometry work (native geo values cannot be hashed by Spark 4.1).

Each section mirrors a named integration test in `havasu-iceberg` (`TestV3GeoSedonaLifecycle`, `TestV3GeoGateOffRoundTrip`, `TestLegacyToV3Migration`, `TestV3GeometryFormatGating`), so a green run here reproduces the release test surface on real staging infrastructure.

## 1. Set up the Sedona context

The two session flags go on the Spark config; the V3-write flag is a **catalog property**, set here client-side on the `examples_temp` catalog. (If the catalog service serves this flag in its `/v1/config` *overrides*, the server value wins — client config only beats the served *defaults*.)

In [ ]:
from sedona.spark import *

config = (
    SedonaContext.builder()
    .config("spark.sql.geospatial.enabled", "true")
    .config("spark.sedona.enableNativeGeoTypes", "false")
    .config("spark.sql.catalog.examples_temp.havasu.geo.v3-write-enabled", "true")
    .getOrCreate()
)
sedona = SedonaContext.create(config)

In [ ]:
# sanity: confirm the session flags are what this notebook assumes
assert sedona.conf.get("spark.sql.geospatial.enabled") == "true"
assert sedona.conf.get("spark.sedona.enableNativeGeoTypes") == "false"
print("flags OK")

In [ ]:
sedona.sql("CREATE DATABASE IF NOT EXISTS examples_temp.iceberg_v3_db")
DB = "examples_temp.iceberg_v3_db"
SRC = f"{DB}.geo_src"
DERIVED = f"{DB}.geo_derived"
CTAS = f"{DB}.geo_ctas"
LEGACY = f"{DB}.geo_legacy"
MIGRATED = f"{DB}.geo_migrated"
for t in [SRC, DERIVED, CTAS, LEGACY, MIGRATED]:
    sedona.sql(f"DROP TABLE IF EXISTS {t}")

## 2. Create a V3 table with typed DDL

`GEOMETRY(4326)` declares the column CRS. Note there is **no explicit `format-version`**: a v3-write-enabled catalog defaults it to 3 when the schema has a native geo column — we verify that below.

*Mirrors: `TestV3GeoSedonaLifecycle.fullLifecycle` step 1, `TestV3GeometryFormatGating.testNativeDdlRequiresFormatVersion3`.*

In [ ]:
sedona.sql(f"CREATE TABLE {SRC} (id BIGINT, geom GEOMETRY(4326)) USING iceberg")
props = sedona.sql(f"SHOW TBLPROPERTIES {SRC}").collect()
fv = [r for r in props if r['key'] == 'format-version']
print(fv)
assert fv[0]['value'] == '3', 'expected the catalog to default format-version to 3'

With the gate **off**, the V3-native column surfaces as the Sedona UDT (`geometry`), not Spark's parameterized `geometry(4326)`:

In [ ]:
sedona.sql(f"DESCRIBE {SRC}").show(truncate=False)

## 3. Write through Sedona ST_ functions

The values are produced by Sedona's own constructors. Gate-off they are UDT-typed, and the writer recodes them to the column's ISO-WKB — validating each value's SRID against the column CRS on the way in.

*Mirrors: `fullLifecycle` step 2.*

In [ ]:
sedona.sql(f"""
INSERT INTO {SRC}
SELECT id, ST_SetSRID(ST_GeomFromText(wkt), 4326) FROM VALUES
  (1L, 'POINT (30 10)'),
  (2L, 'POINT (30 10)'),
  (3L, 'POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0))'),
  (4L, CAST(NULL AS STRING)) AS t(id, wkt)
""")
sedona.sql(f"SELECT COUNT(*) FROM {SRC}").show()

## 4. Read back and verify values + SRIDs

*Mirrors: `fullLifecycle` step 3, `TestV3GeoGateOffRoundTrip.schemaAndValuesAgreeInEveryGateState`.*

In [ ]:
sedona.sql(f"SELECT id, ST_AsText(geom) AS wkt, ST_SRID(geom) AS srid FROM {SRC} ORDER BY id").show(truncate=False)
row = sedona.sql(f"SELECT ST_AsText(geom) w, ST_SRID(geom) s FROM {SRC} WHERE id = 1").collect()[0]
assert row['w'] == 'POINT (30 10)' and row['s'] == 4326
print('round trip OK')

## 5. The ST_ surface binds: measures, predicates, joins

*Mirrors: `fullLifecycle` step 4.*

In [ ]:
area = sedona.sql(f"SELECT ST_Area(geom) a FROM {SRC} WHERE id = 3").collect()[0]['a']
assert area == 16.0, area

dist = sedona.sql(f"""
SELECT ST_Distance(a.geom, b.geom) d
FROM {SRC} a JOIN {SRC} b ON a.id = 1 AND b.id = 3
""").collect()[0]['d']
assert dist > 0.0

hits = sedona.sql(f"""
SELECT COUNT(*) c FROM {SRC}
WHERE ST_Intersects(geom, ST_GeomFromText('POLYGON ((0 0, 5 0, 5 5, 0 5, 0 0))'))
""").collect()[0]['c']
assert hits == 1
print(f'area={area}, distance={dist:.3f}, intersect hits={hits}')

## 6. GROUP BY / DISTINCT on a geometry column

This is the reason the gate-off path exists: Spark 4.1 cannot hash native geo values, so these operations die mid-shuffle with the gate **on** (`scala.MatchError`, unfixed upstream — DB-487). Reading V3 as UDTs keeps native values out of the shuffle entirely.

*Mirrors: `TestV3GeoGateOffRoundTrip.groupByAndDistinctSucceedOnV3GeometryWhenGateIsOff`.*

In [ ]:
distinct = sedona.sql(f"SELECT COUNT(DISTINCT geom) c FROM {SRC}").collect()[0]['c']
assert distinct == 2, distinct
sedona.sql(f"SELECT ST_AsText(geom) wkt, COUNT(*) n FROM {SRC} WHERE geom IS NOT NULL GROUP BY geom ORDER BY n").show(truncate=False)

## 7. Derived results land in a new V3 table

The V3 route for derived tables is **typed DDL + `INSERT`**: the target declares its CRS, then the ST_-produced UDT values are recoded on insert.

*Mirrors: `fullLifecycle` step 5.*

In [ ]:
sedona.sql(f"CREATE TABLE {DERIVED} (id BIGINT, geom GEOMETRY(4326)) USING iceberg")
sedona.sql(f"""
INSERT INTO {DERIVED}
SELECT id, ST_SetSRID(ST_Centroid(geom), 4326) FROM {SRC} WHERE geom IS NOT NULL
""")
c = sedona.sql(f"SELECT ST_AsText(geom) w FROM {DERIVED} WHERE id = 3").collect()[0]['w']
assert c == 'POINT (2 2)', c
print('centroid table OK:', c)

## 8. Know the CTAS nuance: UDT provenance creates a *legacy* table

The catalog flag **permits** V3, it does not force it. A plain CTAS from an ST_-derived frame is `GeometryUDT`-typed, which carries no CRS for a V3 column to adopt — so the created column is Havasu-**legacy** (and the table is not format-version 3). It still reads fine, but it is not V3 storage.

*Mirrors: `TestV3GeoSedonaLifecycle.ctasFromStDerivedFrameCreatesHavasuLegacy`.*

In [ ]:
sedona.sql(f"""
CREATE TABLE {CTAS} USING iceberg AS
SELECT id, ST_Buffer(geom, 1.0) AS geom FROM {SRC} WHERE geom IS NOT NULL
""")
props = {r['key']: r['value'] for r in sedona.sql(f"SHOW TBLPROPERTIES {CTAS}").collect()}
print('format-version:', props.get('format-version', '(default, i.e. not 3)'))
assert props.get('format-version') != '3', 'CTAS from UDT provenance should NOT be V3'

## 9. Dataset-wide SRID validation

A value whose SRID disagrees with the column CRS is **rejected by the writer** with actionable guidance — never silently relabelled. Note `SRID 0` is a real, distinct CRS (not an "unset" marker), so untagged geometries need `ST_SetSRID` before they can enter a 4326 column.

*Mirrors: `TestNativeGeoSedonaInterop.testSridMismatchWriteRejected`.*

In [ ]:
try:
    sedona.sql(f"INSERT INTO {SRC} SELECT 99L, ST_SetSRID(ST_GeomFromText('POINT (1 1)'), 3857)")
    raise AssertionError('mismatched SRID was accepted — should have been rejected')
except Exception as e:
    msg = str(e)
    assert 'SRID 3857' in msg and 'ST_Transform' in msg, msg[:500]
    print('rejected as expected:', msg.split('.')[0])

# nothing was written
assert sedona.sql(f"SELECT COUNT(*) c FROM {SRC} WHERE id = 99").collect()[0]['c'] == 0

## 10. Migrate a legacy table to V3

Legacy tables (created from `GeometryUDT` DataFrames — every pre-V3 pipeline) migrate with the recipe: **pre-flight the SRIDs, typed-DDL target, `INSERT ... SELECT`**. There is no in-place `ALTER COLUMN` from legacy to V3.

*Mirrors: `TestLegacyToV3Migration.migrateLegacyGeometryTableToV3`.*

In [ ]:
from sedona.sql.types import GeometryType
from pyspark.sql.types import StructType, StructField, LongType
from shapely import wkt as shapely_wkt

schema = StructType([StructField('id', LongType(), False), StructField('geom', GeometryType(), True)])
rows = [(1, shapely_wkt.loads('POINT (30 10)')), (2, shapely_wkt.loads('POINT (-71 42)'))]
sedona.createDataFrame(rows, schema).writeTo(LEGACY).create()

# legacy provenance: even with the flag on, UDT provenance creates a legacy (non-V3) table
props = {r['key']: r['value'] for r in sedona.sql(f'SHOW TBLPROPERTIES {LEGACY}').collect()}
assert props.get('format-version') != '3'
print('legacy table created, format-version:', props.get('format-version', '(default)'))

In [ ]:
# pre-flight: one SRID? (0 here — shapely geometries carry no SRID)
sedona.sql(f"SELECT DISTINCT ST_SRID(geom) FROM {LEGACY}").show()

# migrate: typed DDL + INSERT, labelling the SRID explicitly
sedona.sql(f"CREATE TABLE {MIGRATED} (id BIGINT, geom GEOMETRY(4326)) USING iceberg")
sedona.sql(f"INSERT INTO {MIGRATED} SELECT id, ST_SetSRID(geom, 4326) FROM {LEGACY}")

props = {r['key']: r['value'] for r in sedona.sql(f'SHOW TBLPROPERTIES {MIGRATED}').collect()}
assert props.get('format-version') == '3'
sedona.sql(f"SELECT id, ST_AsText(geom) wkt, ST_SRID(geom) srid FROM {MIGRATED} ORDER BY id").show(truncate=False)

## 11. Optional: what the gate-on view looks like

The same tables, read with `spark.sedona.enableNativeGeoTypes=true`, surface Spark's **native** `geometry(4326)` type — storage did not change, only the in-memory representation. 

> ⚠️ Gate-on, avoid `GROUP BY`/`DISTINCT`/joins **on geometry columns** — Spark 4.1 cannot hash native geo values (DB-487).

*Mirrors: `TestV3GeoGateOffRoundTrip.v3GeoStaysNativeWhenGateIsOn`.*

In [ ]:
sedona.conf.set('spark.sedona.enableNativeGeoTypes', 'true')
sedona.sql(f'DESCRIBE {SRC}').show(truncate=False)   # geom is now geometry(4326)
sedona.sql(f'SELECT id, geom FROM {SRC} ORDER BY id').show(truncate=False)
sedona.conf.set('spark.sedona.enableNativeGeoTypes', 'false')  # restore the notebook's mode

## 12. REST `createOrReplace` probe (DB-473 Defect 2)

This staging run is the first time the gate-off V3 flows meet the **real Wherobots REST catalog** — and the one unverified risk from DB-473 lives exactly here. Its Defect 2: a gate-off `createOrReplace` on a table with V3 geo history failed at commit with

```
CommitFailedException: Cannot set last added schema: no schema has been added
```

and it only reproduced on the **REST** commit path (Hadoop-catalog controls all passed), so none of the local integration suites can settle it. This section runs both replace shapes and *classifies* the outcome — a `Cannot set last added schema` failure here is the known, tracked DB-473 Defect 2 resurfacing, **not** a notebook bug.

**Probe A** — SQL `CREATE OR REPLACE` restating the V3 schema: allowed by the DB-224 gate (it introduces no new V3 column) and commits a same-schema replace through REST.

In [ ]:
COR = f"{DB}.geo_cor"
sedona.sql(f"DROP TABLE IF EXISTS {COR}")
sedona.sql(f"CREATE TABLE {COR} (id BIGINT, geom GEOMETRY(4326)) USING iceberg")
sedona.sql(f"INSERT INTO {COR} SELECT 1L, ST_SetSRID(ST_GeomFromText('POINT (30 10)'), 4326)")

try:
    sedona.sql(f"CREATE OR REPLACE TABLE {COR} (id BIGINT, geom GEOMETRY(4326)) USING iceberg")
    n = sedona.sql(f'SELECT COUNT(*) c FROM {COR}').collect()[0]['c']
    print(f'Probe A PASSED: same-schema V3 replace committed on REST (rows after replace: {n})')
except Exception as e:
    if 'Cannot set last added schema' in str(e):
        print('Probe A: DB-473 Defect 2 REPRODUCED on the REST commit path — report on DB-473, this is the tracked failure')
    else:
        raise

**Probe B** — the exact DB-473 Defect 2 repro: a gate-off `GeometryUDT` DataFrame `createOrReplace` onto the table with V3 schema history. The UDT provenance derives a *legacy* schema, so this replace flips the table's current schema from V3 to legacy over REST — the mixed-schema-history shape that produced the original commit failure.

In [ ]:
udt_df = sedona.createDataFrame(
    [(10, shapely_wkt.loads('POINT (5 5)'))],
    StructType([StructField('id', LongType(), False), StructField('geom', GeometryType(), True)]),
)

try:
    udt_df.writeTo(COR).createOrReplace()
    n = sedona.sql(f'SELECT COUNT(*) c FROM {COR}').collect()[0]['c']
    assert n == 1, f'replace committed but row count wrong: {n}'
    print('Probe B PASSED: UDT createOrReplace over V3 history committed on REST — DB-473 Defect 2 did not reproduce')
except Exception as e:
    if 'Cannot set last added schema' in str(e):
        print('Probe B: DB-473 Defect 2 REPRODUCED on the REST commit path — report on DB-473, this is the tracked failure')
    else:
        raise

## 13. Cleanup

Uncomment to drop everything this notebook created.

In [ ]:
# for t in [SRC, DERIVED, CTAS, LEGACY, MIGRATED, COR]:
#     sedona.sql(f'DROP TABLE IF EXISTS {t}')
# sedona.sql('DROP DATABASE IF EXISTS examples_temp.iceberg_v3_db')